In [ ]:
import torch
from botorch.exceptions import InputDataWarning

from bott.problem import OptimizationProblem
from bott.physics_models import simulate_cbed
from bott.optimization import run_one_trial

import warnings
warnings.filterwarnings("ignore", category=InputDataWarning) 
# InputDataWarning: Data (outcome observations) is not standardized (std = tensor([5.3673e-11, 5.0607e-10, 4.8782e-10, 7.7036e-11, 5.1824e-14],
#        dtype=torch.float64), mean = tensor([0.0000e+00, 0.0000e+00, 8.4703e-22, 0.0000e+00, 0.0000e+00],
#        dtype=torch.float64)).Please consider scaling the input to zero mean and unit variance.
#   check_standardization(Y=train_Y, raise_on_fail=raise_on_fail)

In [ ]:
ground_truth = torch.Tensor(simulate_cbed(20,10,-10, device_simu='gpu')) # abtem takes "cpu" or "gpu"

In [ ]:
# OptimizationProblem would keep all the tensor on the specified device
problem = OptimizationProblem(ground_truth=ground_truth,
                              output_path='./output', 
                              save_results=True, 
                              reduction_params={'reduction_type':'square', 'reduction_kwargs':{'num_tiles':2}},
                              loss_params={'loss_type':'SSE', 'dp_pow': 0.5}, 
                              norm_arr=False,
                              dim=3, 
                              bounds=[(15,25), (-20, 20), (-20, 20)],
                              noise_std=0,
                              dtype=torch.float64, 
                              device='cuda'
                              ) # "cpu" or "cuda" for physics simulation

In [ ]:
run_one_trial(problem_name='EICF', 
              problem=problem, 
              algo='EICF', 
              trial=3, 
              n_init_evals=2, 
              max_iter=500, 
              objective=None,
              dtype=torch.float64,
              device_botorch='cuda'
              )

In [ ]:
from botorch.acquisition.objective import GenericMCObjective

# Initialize the problem = OptimizationProblem(...)

loss_func = problem.loss_func
reduction_true = problem.reduction_true
Y = torch.rand([3,5]).to('cuda') # [n_samples, n_total_tiles+1]

def wrap_func(Y, X=None):
    return -1*loss_func(Y[...,:-1], reduction_true, reduce=False) + Y[...,-1]

objective0 = GenericMCObjective(lambda Y, X=None: -1*(loss_func(Y[...,:-1], reduction_true) + Y[...,[-1]]).squeeze(-1)) # Pass
objective1 = GenericMCObjective(lambda Y, X=None: -1*(loss_func(Y[...,:-1], reduction_true, reduce=False) + Y[...,-1])) # RuntimeError: The q-batch shape of the objective values does not agree with the q-batch shape of X.
objective2 = GenericMCObjective(lambda Y, X=None: -1*((Y[...,:-1] - reduction_true).pow(2).sum(-1) + Y[...,-1])) # Pass
objective3 = GenericMCObjective(wrap_func)

print(objective0(Y).shape)
print(objective1(Y).shape)
print(objective2(Y).shape)
print(objective3(Y).shape)

run_one_trial(problem_name='EICF', 
              problem=problem, 
              algo='EICF', 
              trial=3, 
              n_init_evals=2, 
              max_iter=500, 
              objective=objective2,
              dtype=torch.float64,
              device_botorch='cuda'
              )